# Explore the Amazon `description` and `features` columns

**Aim:** Understand the text available alongside our prepared table of contents, so we can choose a simple first input for recommendations.

`description` and `features` are **two separate columns in `data/raw/books.parquet`**. Both come from the saved Amazon book record. Here, `features` means that actual source column. [Notebook 03](03_toc_preparation_preview.ipynb)'s prepared TOC text is a separate dataset.

One row represents one Amazon book record, identified by `parent_asin`. The 9,034 saved records are ordered by that ID and are not a random sample of all books. All findings below describe this local snapshot.

| Section | What we learn or validate |
| --- | --- |
| [1. Load and inspect the columns](#section-1) | Check storage and piece types, then see the first ten elements of both lists for one book. |
| [2. Read one book](#section-2) | Read the complete lists for the book selected in section 1. |
| [3. Measure availability](#section-3) | See how pieces become one string, then count books with text in either column, both, or neither. |
| [4. Compare the two texts](#section-4) | Check whether joining both columns would repeat an entire field. |
| [5. Measure length](#section-5) | See typical lengths and identify unusually short or long values. |
| [6. Decide what to try first](#section-6) | State the next step; verify every original piece, its order, and the raw file's checksum. |

Run the cells in order using the project's `.venv` kernel. This notebook only reads the saved file; its joined text exists in memory for inspection.

<a id="section-1"></a>

## 1. What do the two columns store?

**Objective:** Load the book ID, title, `description`, and `features`, then check the shape of their saved values.

The file checksum lets us confirm at the end that the raw file has not changed.

In [1]:
import hashlib
import html
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the BookPath repository.")
    project_root = project_root.parent

raw_path = project_root / "data/raw/books.parquet"


def file_checksum(path):
    with path.open("rb") as saved_file:
        return hashlib.file_digest(saved_file, "sha256").hexdigest()


checksum_before = file_checksum(raw_path)
text_columns = ["description", "features"]
books = pd.read_parquet(raw_path, columns=["parent_asin", "title", *text_columns])
assert books["parent_asin"].notna().all() and books["parent_asin"].is_unique
print(f"Loaded {len(books):,} book records, each with a unique parent_asin.")
print("Columns loaded:", ", ".join(books.columns))

Loaded 9,034 book records, each with a unique parent_asin.
Columns loaded: parent_asin, title, description, features


Each cell holds an **array of text pieces**, which behaves like a list. A piece can be a heading, a paragraph, or even part of a sentence. It is not necessarily a bullet point or a chapter.

In the table below, **Saved pieces** counts all elements inside those arrays. **Blank pieces** counts strings containing only spaces; **Non-text pieces** counts elements that are not strings. Empty arrays contain zero pieces and are counted separately in section 3.

In [2]:
storage_rows = []
for column in text_columns:
    pieces = [piece for value in books[column] for piece in value]
    storage_rows.append({
        "Column in books": column,
        "Saved pieces": len(pieces),
        "Blank pieces": sum(isinstance(piece, str) and not piece.strip() for piece in pieces),
        "Non-text pieces": sum(not isinstance(piece, str) for piece in pieces),
    })

storage = pd.DataFrame(storage_rows).set_index("Column in books")
display(storage)
assert storage["Non-text pieces"].eq(0).all(), "Inspect non-text pieces before joining."
print(f"Answer: all {storage['Saved pieces'].sum():,} saved pieces are strings; "
      f"{storage['Blank pieces'].sum():,} are blank.")


,Saved pieces,Blank pieces,Non-text pieces
Column in books,,,
description,253557,0,0
features,71241,0,0


Answer: all 324,798 saved pieces are strings; 0 are blank.


**What do the two lists look like?** The table above counts pieces across all books. Below, we print the **first 10 elements of each list for one book**, in their saved order, with brackets and quotes showing where each element begins and ends. No text is trimmed or joined in this preview. If a chosen list has fewer than 10 elements, we show all of them.

Start with *Never Split the Difference*, `parent_asin = "0062407805"`; both lists have at least 10 elements. Change `example_asin` to another existing ID, or uncomment the random choice. Keeping `random_state=42` repeats the same choice. After changing it, rerun this cell and sections 2–3 to refresh the full lists and joining example. This example illustrates storage, not typical content across all books.


In [3]:
example_asin = "0062407805"
# example_asin = books.sample(n=1, random_state=42)["parent_asin"].iloc[0]
selected = books.loc[books["parent_asin"].eq(example_asin)]
if selected.empty:
    raise ValueError(f"No saved book has parent_asin = {example_asin}.")
selected_book = selected.iloc[0]
print(f"{selected_book['title']} (parent_asin = {example_asin})")

for column in text_columns:
    pieces = list(selected_book[column])
    print(f"\n{column}: showing the first {min(10, len(pieces))} of {len(pieces):,} saved elements:")
    print(json.dumps(pieces[:10], ensure_ascii=False, indent=2))

Never Split the Difference: Negotiating As If Your Life Depended On It (parent_asin = 0062407805)

description: showing the first 10 of 35 saved elements:
[
  "Review",
  "Former FBI Hostage Negotiator Chris Voss has few equals when it comes to high stakes negotiations. Whether for your business or your personal life, his techniques work.” —",
  "Joe Navarro, FBI Special Agent (Ret.) and author of the international bestseller,",
  "What Every Body is Saying",
  "Chatty and friendly and packed with helpful resources, this is an intriguing approach to business and personal negotiations. —",
  "Publishers Weekly",
  "From the Back Cover",
  "A field-tested, game-changing approach to high-stakes negotiations—whether in the boardroom or at home.",
  "Never Split the Difference",
  "is a riveting, indispensable handbook of negotiation principles culled and perfected from Chris Voss’s remarkable career as a hostage negotiator and later as an award-winning teacher in the world’s most prestigio

<a id="section-2"></a>

## 2. What does one book look like?

**Objective:** Read the complete saved lists for the book selected in section 1, before combining or comparing them.

Section 1 shows the first 10 elements from each list. Expand either list below to read **all its elements**, in their original order. To choose a different book, change `example_asin` in section 1 and rerun that preview and sections 2–3.


In [4]:
# HTML details make long values expandable; escaping keeps the source text literal.
for column in text_columns:
    pieces = list(selected_book[column])
    list_items = "".join(f"<li>{html.escape(piece)}</li>" for piece in pieces)
    content = f"<ol>{list_items}</ol>" if pieces else "<p>This array is empty.</p>"
    display(HTML(
        f"<details><summary>Read all {len(pieces)} pieces from {column}</summary>"
        f"<div style='max-width: 90ch; white-space: pre-wrap'>{content}</div></details>"
    ))

**For the default book:** `features` contains a sales summary explaining the book's negotiation approach. `description` includes reviews, back-cover text, and author biographies. The two column names do not guarantee two different kinds of content.

**Why the pieces matter:** A title or sentence may span several pieces. We will keep their order when joining them. Repeated short pieces can be valid parts of the prose, so we have not removed them. A different selected book may contain different material.

<a id="section-3"></a>

## 3. How many books have text?

**Objective:** Count books with nonblank text in each column and measure the extra coverage from using both.

We create `text_view`, an inspection copy with one string per book and column. Each saved piece is trimmed at its ends, then joined to the next with a line break. The original arrays in `books` stay intact.

**With text** means at least one nonblank piece. It measures presence, not whether the content is useful for recommendations. Every percentage in this section uses **all book records** as its denominator. One book can have text in both columns, so the first table's rows overlap.

In [5]:
def join_text_pieces(pieces):
    return "\n".join(piece.strip() for piece in pieces if piece.strip())


text_view = books[["parent_asin", "title"]].copy()
for column in text_columns:
    text_view[column] = books[column].map(join_text_pieces)

has_text = text_view[text_columns].ne("")
coverage = pd.DataFrame({
    "With text": has_text.sum(),
    "Without text": (~has_text).sum(),
    "With text (%)": (100 * has_text.mean()).round(2),
})
coverage.index.name = "Column in books"
display(coverage)
print(f"Answer: features has text for {coverage.loc['features', 'With text']:,} books; "
      f"description has text for {coverage.loc['description', 'With text']:,}.")

,With text,Without text,With text (%)
Column in books,,,
description,8006,1028,88.62
features,8991,43,99.52


Answer: features has text for 8,991 books; description has text for 8,006.


**Before and after joining:** For the book selected in section 1, we show the first three pieces of each column, then the single string they produce. This short preview uses the same `join_text_pieces()` function used above for every complete list.

The function trims whitespace at each piece's ends, skips blank pieces, and inserts a line break between the remaining pieces. Their order stays the same; repeated pieces stay too. In Python, `"\n"` means a line break. `description` and `features` are joined separately into their own strings in `text_view`. The original lists in `books` stay intact.


In [6]:
print(f"Joining example: {selected_book['title']} (parent_asin = {example_asin})")
for column in text_columns:
    preview_pieces = list(selected_book[column][:3])
    print(f"\n{column} — before: first {len(preview_pieces)} saved pieces")
    print(json.dumps(preview_pieces, ensure_ascii=False, indent=2))
    print("After: one string, with a line break between pieces")
    joined_preview = join_text_pieces(preview_pieces)
    print(joined_preview if joined_preview else "(empty string)")

print("Conclusion: each column becomes its own string; nonblank pieces keep their order. "
      "Only this preview is limited to three pieces; text_view uses the complete lists.")


Joining example: Never Split the Difference: Negotiating As If Your Life Depended On It (parent_asin = 0062407805)

description — before: first 3 saved pieces
[
  "Review",
  "Former FBI Hostage Negotiator Chris Voss has few equals when it comes to high stakes negotiations. Whether for your business or your personal life, his techniques work.” —",
  "Joe Navarro, FBI Special Agent (Ret.) and author of the international bestseller,"
]
After: one string, with a line break between pieces
Review
Former FBI Hostage Negotiator Chris Voss has few equals when it comes to high stakes negotiations. Whether for your business or your personal life, his techniques work.” —
Joe Navarro, FBI Special Agent (Ret.) and author of the international bestseller,

features — before: first 3 saved pieces
[
  "A former international hostage negotiator for the FBI offers a new, field-tested approach to high-stakes negotiations―whether in the boardroom or at home.",
  "** A",
  "Wall Street Journal"
]
After: one

**Does using both reach more books?** The next table places each book in exactly one group. Its four groups add up to the **Total** row.

For example, **Only features** means the book's `features` column contains text but its `description` column does not.

In [7]:
availability = pd.Series({
    "Both columns": (has_text["description"] & has_text["features"]).sum(),
    "Only features": (~has_text["description"] & has_text["features"]).sum(),
    "Only description": (has_text["description"] & ~has_text["features"]).sum(),
    "Neither column": (~has_text["description"] & ~has_text["features"]).sum(),
}, name="Book records").to_frame()
assert availability["Book records"].sum() == len(books)
availability["All books (%)"] = (100 * availability["Book records"] / len(books)).round(2)
availability.loc["Total"] = [len(books), 100.0]
availability["Book records"] = availability["Book records"].astype(int)
display(availability)

print(f"Answer: description supplies text for {availability.loc['Only description', 'Book records']:,} "
      "additional books beyond features.")
print(f"Together they cover {has_text.any(axis=1).sum():,} of {len(books):,} books; "
      f"{availability.loc['Neither column', 'Book records']:,} have neither.")

,Book records,All books (%)
Both columns,7996,88.51
Only features,995,11.01
Only description,10,0.11
Neither column,33,0.37
Total,9034,100.00


Answer: description supplies text for 10 additional books beyond features.
Together they cover 9,001 of 9,034 books; 33 have neither.


<a id="section-4"></a>

## 4. Would combining the columns repeat text?

**Objective:** Among books with text in both columns, check whether one whole text is already inside the other.

For this comparison only, we ignore uppercase/lowercase and replace runs of whitespace with one space. We keep punctuation and word order. **Contained** means an exact substring match after those changes; this does not measure similarity of meaning or all shared phrases.

Each book belongs to one group, checked in this order: identical texts; description inside features; features inside description; neither whole text contained. The denominator is **books with both columns**, not all books.

In [8]:
comparison = text_view.loc[has_text.all(axis=1), ["parent_asin", *text_columns]].copy()
for column in text_columns:
    comparison[column] = (
        comparison[column].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
    )


def compare_texts(book):
    if book.description == book.features:
        return "Identical texts"
    if book.description in book.features:
        return "Whole description inside features"
    if book.features in book.description:
        return "Whole features inside description"
    return "Neither whole text contained"


comparison["Result"] = [compare_texts(book) for book in comparison.itertuples()]
groups = ["Identical texts", "Whole description inside features",
          "Whole features inside description", "Neither whole text contained"]
overlap = comparison["Result"].value_counts().reindex(groups, fill_value=0).to_frame("Book records")
assert overlap["Book records"].sum() == len(comparison)
overlap["Books with both (%)"] = (100 * overlap["Book records"] / len(comparison)).round(2)
overlap.loc["Total with both"] = [len(comparison), 100.0]
overlap["Book records"] = overlap["Book records"].astype(int)
display(overlap)
print(f"Answer: the whole features text already appears inside description for "
      f"{overlap.loc['Whole features inside description', 'Book records']:,} books.")

,Book records,Books with both (%)
Result,,
Identical texts,0,0.00
Whole description inside features,0,0.00
Whole features inside description,207,2.59
Neither whole text contained,7789,97.41
Total with both,7996,100.00


Answer: the whole features text already appears inside description for 207 books.


**What this means:** Concatenating both columns would repeat the entire `features` text for the books in that containment group. For the remaining books, the fields may still repeat ideas or smaller passages. This simple comparison does not establish that every extra word adds information.

We therefore keep the two sources separate during this EDA. There is no automatic merging or deletion rule.

<a id="section-5"></a>

## 5. How much text would we be adding?

**Objective:** Compare lengths among books with text and show the shortest and longest examples for inspection.

We count **characters in the joined strings**, including the line breaks inserted between pieces. These are not model token counts. Empty values are excluded from this length table; their counts are already in section 3. The median is the middle length, and the 90th percentile describes roughly where nine out of ten lengths fall. Percentiles below are rounded to whole characters.

In [9]:
lengths = text_view[text_columns].apply(lambda column: column.str.len()).where(has_text)
length_summary = pd.DataFrame({
    "Books with text": lengths.count(),
    "Shortest (characters)": lengths.min(),
    "Median (characters)": lengths.median(),
    "90th percentile (characters)": lengths.quantile(0.9),
    "Longest (characters)": lengths.max(),
}).round().astype(int)
length_summary.index.name = "Column in books"
display(length_summary)
print(f"Answer: median features length is {length_summary.loc['features', 'Median (characters)']:,} "
      f"characters; median description length is {length_summary.loc['description', 'Median (characters)']:,}.")

,Books with text,Shortest (characters),Median (characters),90th percentile (characters),Longest (characters)
Column in books,,,,,
description,8006,16,2776,14022,300981
features,8991,1,1278,2368,22496


Answer: median features length is 1,278 characters; median description length is 2,776.


**Read the extremes in context.** The next table identifies one shortest and one longest nonempty value per column and shows its first 160 characters. Set `example_asin` in section 1 to one of these IDs to read the complete arrays. These four selected examples are not a representative sample.

Length alone cannot tell us whether text is useful. In particular, a one-character value counted as present still needs interpretation.

In [10]:
length_examples = []
for column in text_columns:
    for label, index in [("Shortest", lengths[column].idxmin()), ("Longest", lengths[column].idxmax())]:
        book = text_view.loc[index]
        length_examples.append({
            "Column": column,
            "Example": label,
            "parent_asin": book["parent_asin"],
            "Characters": int(lengths.loc[index, column]),
            "Beginning (up to 160 characters)": book[column][:160],
        })

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(length_examples))

,Column,Example,parent_asin,Characters,Beginning (up to 160 characters)
0,description,Shortest,142213167X,16,About the Author
1,description,Longest,0684852861,300981,"Amazon.com Review\nMarcus Buckingham and Curt Coffman expose the fallacies of standard management thinking in\nFirst, Break All the Rules: What the World's Greate"
2,features,Shortest,0961413913,1,n
3,features,Longest,0470592206,22496,"A practical guide to passive investing\nTime and again, individual investors discover, all too late, that actively picking stocks is a loser's game. The alternat"


**Answer:** The shortest `features` value is only `"n"`; the shortest `description` is `"About the Author"`. Neither explains what the book teaches. At the other extreme, one description contains 300,981 characters. These examples show why nonblank text and a useful book summary are different things. We retain the values and record this limitation. No length cutoff has been applied.


<a id="section-6"></a>

## 6. What should we try first?

**Objective:** Use the evidence above to bound the next implementation step.

- **`features` is the first candidate for book-level text alongside the prepared TOC:** it reaches more books and has a shorter median length than `description`. Our example shows that it can describe what the book teaches. Coverage and length alone do not establish recommendation quality.
- **Keep `description` available:** it supplies text for the books with an empty `features` value, and may contain additional useful material for other books. Reviews, author biographies, and long excerpts also occur, so adding all of it is not automatically better.
- **Keep each source identifiable:** whole-field containment is enough to show that indiscriminate concatenation can repeat text. No deduplication or length cutoff has been chosen here.

**Stopping point:** This first pass establishes storage, availability, examples, overlap, and length. The next implementation can compare the prepared TOC alone with TOC plus book-level text. Further cleaning should respond to observed problems in that working version.

[Notebook 02](02_toc_eda.ipynb#section-8)'s possible chapter lists in these columns remain a separate recovery opportunity. We have not extracted or merged them into the TOC.

**Final validation — were the original lists preserved?** We reread `parent_asin`, `description`, and `features` from the saved Parquet file. For every book, we compare its ID and every piece in each list, including the pieces' order. `map(list)` puts both versions into ordinary lists for comparison; it does not trim or join their text.

`assert_series_equal()` stops the notebook if a piece is edited, added, removed, reordered, or attached to a different book. We also check the coverage totals and the raw file's checksum. The success message appears only after every check passes. This validates preservation of the source data; it does not establish that the content will improve recommendations.

In [11]:
assert coverage["With text"].add(coverage["Without text"]).eq(len(books)).all()
saved_books = pd.read_parquet(raw_path, columns=["parent_asin", *text_columns])
pd.testing.assert_series_equal(books["parent_asin"], saved_books["parent_asin"], check_exact=True)
for column in text_columns:
    assert books[column].map(len).sum() == storage.loc[column, "Saved pieces"]
    pd.testing.assert_series_equal(
        books[column].map(list), saved_books[column].map(list),
        check_exact=True, obj=f"{column}: original pieces and their order",
    )

assert file_checksum(raw_path) == checksum_before, "The raw file changed during inspection."
print(f"Conclusion: all {storage['Saved pieces'].sum():,} original pieces across "
      f"both columns and {len(books):,} books match the saved file exactly, in the same order. "
      "Coverage checks passed; raw file unchanged.")
print("Source SHA-256:", checksum_before)


Conclusion: all 324,798 original pieces across both columns and 9,034 books match the saved file exactly, in the same order. Coverage checks passed; raw file unchanged.
Source SHA-256: 4acf6979b15215c28f2a45a1488ace4d749e2573f0fa4a0110e2933dd0194dd8
